## Syn Bank Wallet Intelligence — Pipeline

Reproducible pipeline from raw data to the consolidated client table. Cleaning and 
aggregation rules applied here were derived and justified in `01_eda.ipynb` — see that 
notebook for the reasoning behind each decision.

**Scope:** analysis covers the 20 clients (E01–E20) present in the provided datasets. 
The brief describes a 50-client portfolio; no data was provided for the remaining 30 
(E21–E50), so this pipeline and all downstream outputs are scoped to the 20 observed 
clients.

In [1]:
import pandas as pd
pd.options.display.float_format = '{:,.2f}'.format

cross_border_payments = pd.read_csv("../data/cross_border_payments.csv")
trade_finance = pd.read_csv("../data/trade_finance.csv")
transactional_banking = pd.read_csv("../data/transactional_banking.csv")

# Currency casing fix (found during EDA: 'ZAR' vs 'zar')
transactional_banking['currency'] = transactional_banking['currency'].str.upper()

print(cross_border_payments.shape, transactional_banking.shape, trade_finance.shape)

(241117, 13) (2802875, 13) (20303, 15)


## Consolidated per-client table

In [2]:
cb = cross_border_payments.copy()
tb = transactional_banking.copy()
cb['date'] = pd.to_datetime(cb['date'])
tb['date'] = pd.to_datetime(tb['date'])

matches_list = []

# TODO: ask christiaan about this day offsett thing
# #TODO: could add a section in limitation and assumptions - shows we are thinking about real world applic and risk of false postives.
# internation SWIFT payment takes a few days to clear
# create a week long "lag" to account for this delay
for day_offset in range(-3, 4):
    # subset columns for memory effiecient
    cb_shifted = cb[['transaction_id', 'entity_id', 'date', 'value_zar', 'direction']].copy()
    # only shift the cross border dataset
    cb_shifted['join_date'] = cb_shifted['date'] + pd.Timedelta(days=day_offset)
    
    # leave transactional banking dataset "join date" as is
    tb_subset = tb[['transaction_id', 'entity_id', 'date', 'amount_zar', 'direction']].copy()
    tb_subset['join_date'] = tb_subset['date']
    
    # perform inner join - keeps rows where "on" columns are the same
    merged = pd.merge(
        cb_shifted,
        tb_subset,
        on=['entity_id', 'direction', 'join_date'],
        suffixes=('_cb', '_tb')
    )
    
    # filter out amounts within 1% margin (allow for processing fees, rounding errors and fx spreads)
    merged['amount_diff_pct'] = ((merged['value_zar'] - merged['amount_zar']).abs() / merged['value_zar']) * 100
    matched = merged[merged['amount_diff_pct'] <= 1.0].copy()
    
    if len(matched) > 0:
        matched['day_diff'] = abs(day_offset)
        matches_list.append(matched)

# concat all merged matches
if matches_list:
    # SUPER important. might have mutiple transaction of the exam same near same date.
    # so loop matches multiple transactions to one another.. need to drop those
    overlap_check = pd.concat(matches_list, ignore_index=True)
    
    # sort by the smallest date difference, then the smallest ZAR amount variance
    overlap_check = overlap_check.sort_values(by=['day_diff', 'amount_diff_pct'])
    
    # drop duplicates keeping only the FIRST (best) match for each Cross-Border ID
    overlap_check = overlap_check.drop_duplicates(subset=['transaction_id_cb'], keep='first')
    
    # ensure no Transactional ID was double-booked either
    overlap_check = overlap_check.drop_duplicates(subset=['transaction_id_tb'], keep='first')
else:
    # fallback empty dataframe if no matches exist
    overlap_check = pd.DataFrame(columns=['transaction_id_cb', 'transaction_id_tb'])

print(f"Total overlapping cross-ledger transactions to deduplicate: {len(overlap_check)}")

Total overlapping cross-ledger transactions to deduplicate: 125344


The length being 125 344 actually makes sense, in intial eda crossborder dataset split had "intercompany" at 107 969 rows and "trade" and "other" combined as 133 148 so 125 344 is a solid fuzzy match

In [ ]:
# check which cross-border categories are driving the overlap
validation = pd.merge(
    overlap_check, 
    cross_border_payments[['transaction_id', 'corridor_type']], 
    left_on='transaction_id_cb', 
    right_on='transaction_id', 
    how='left'
)
print(validation['corridor_type'].value_counts())

corridor_type
intercompany    56896
trade           56747
other           12446
Name: count, dtype: int64


In [ ]:
# list of clients bare info
clients = (
    transactional_banking[['entity_id', 'entity_name', 'sector']]
    .drop_duplicates()
    .sort_values('entity_id')
    .reset_index(drop=True)
)

# "intercompany_sweeps" - automated internal treasurary movements 
#   shouldn't be apart of wallet share
# "memo" - hidden lending and facility drawdowns
#   should be removed to not incorrectly categorize loan as cash management
tb_lending_mask = transactional_banking['memo'].notna()
tb_core = transactional_banking[
    (transactional_banking['leg_type'] != 'intercompany_sweeps')
    & (~tb_lending_mask)
]
tb_totals = (
    tb_core.groupby('entity_id')['amount_zar']
    .sum()
    .rename('txn_banking_total_zar')
)

overlapping_cb_ids = set(overlap_check['transaction_id_cb'].unique()) 

# TODO: ask christiaan about removeing "intercompany" 
# removes intercompany, memo tagged rows and overlapping duplicates
cb_lending_mask = cross_border_payments['memo'].notna()
cb_core = cross_border_payments[
    (cross_border_payments['corridor_type'] != 'intercompany')
    & (~cb_lending_mask)
    & (~cross_border_payments['transaction_id'].isin(overlapping_cb_ids))
]
cb_totals = (
    cb_core.groupby('entity_id')['value_zar']
    .sum()
    .rename('cross_border_total_zar')
)

# same logic as before. drop memo
tf_lending_mask = trade_finance['memo'].notna()
tf_core = trade_finance[~tf_lending_mask]
tf_totals = (
    tf_core.groupby('entity_id')['value_zar']
    .sum()
    .rename('trade_finance_total_zar')
)

# of the removed memo rows from last 3 masks, calculate total zar and number of transactions
lending_parts = [
    transactional_banking.loc[tb_lending_mask, ['entity_id', 'amount_zar']]
        .rename(columns={'amount_zar': 'value_zar'}),
    cross_border_payments.loc[cb_lending_mask, ['entity_id', 'value_zar']],
    trade_finance.loc[tf_lending_mask, ['entity_id', 'value_zar']],
]
lending_combined = pd.concat(lending_parts, ignore_index=True)

lending_totals = (
    lending_combined.groupby('entity_id')['value_zar']
    .agg(lending_signal_total_zar='sum', lending_signal_txn_count='count')
)

# attach product totals to clients and create master total
client_table = (
    clients
    .merge(tb_totals, on='entity_id', how='left')
    .merge(cb_totals, on='entity_id', how='left')
    .merge(tf_totals, on='entity_id', how='left')
    .merge(lending_totals, on='entity_id', how='left')
)

value_cols = [
    'txn_banking_total_zar', 'cross_border_total_zar',
    'trade_finance_total_zar', 'lending_signal_total_zar', 'lending_signal_txn_count'
]
client_table[value_cols] = client_table[value_cols].fillna(0)

# cash captured by Syn Bank
client_table['syn_bank_observed_total_zar'] = (
    client_table['txn_banking_total_zar']
    + client_table['cross_border_total_zar']
    + client_table['trade_finance_total_zar']
    + client_table['lending_signal_total_zar']
)

display(client_table.sort_values('syn_bank_observed_total_zar', ascending=False))

,entity_id,entity_name,sector,txn_banking_total_zar,cross_border_total_zar,trade_finance_total_zar,lending_signal_total_zar,lending_signal_txn_count,syn_bank_observed_total_zar
10,E11,Pepkor Holdings,consumer,"47,892,903,881.57","4,880,907,663.25","4,539,648,198.61",0.00,0.00,"57,313,459,743.43"
0,E01,BHP Group,mining,"31,756,486,357.97","2,267,512,966.88","3,949,315,477.88",0.00,0.00,"37,973,314,802.73"
7,E08,Sanlam,insurance,"34,276,672,578.08","2,314,947,457.56","580,026,756.05",0.00,0.00,"37,171,646,791.69"
9,E10,Bid Corporation,consumer,"14,434,456,962.13","5,744,444,137.00","5,461,464,625.72","206,869,183.46",881.00,"25,847,234,908.31"
15,E16,MTN Group,telecoms,"13,471,482,310.31","7,415,844,536.99","4,391,391,161.12","226,999,040.49",844.00,"25,505,717,048.91"
2,E03,Anglo American,mining,"17,951,860,846.36","1,642,429,336.60","2,339,380,305.92",0.00,0.00,"21,933,670,488.88"
8,E09,Shoprite Holdings,consumer,"13,368,347,566.21","3,766,577,869.60","4,260,903,937.99","175,358,263.48",793.00,"21,571,187,637.28"
1,E02,Glencore,mining,"8,304,404,809.97","3,444,080,075.02","3,104,991,006.78","55,002,648.65",83.00,"14,908,478,540.42"
17,E18,The Bidvest Group,industrials_pharma,"5,777,326,306.41","2,144,064,403.02","3,753,096,365.41","101,245,900.56",409.00,"11,775,732,975.40"
18,E19,Aspen Pharmacare,industrials_pharma,"4,105,327,404.36","1,875,412,936.74","1,292,890,353.81","59,876,514.15",408.00,"7,333,507,209.06"
